In [0]:
import re
import json
import random
from datetime import datetime
from cdp_dq_framework.utils import strip_glob

In [0]:
dbutils.fs.rm("/Workspace/GDDT-BDMTODBX/cdp1_framework/notebooks/v2/")

False

In [0]:
# Retrieve job parameters
bucket = dbutils.widgets.get("bucket_name")
file_key = dbutils.widgets.get("bucket_key")
catalog = dbutils.widgets.get("catalog")
schema_nm = dbutils.widgets.get("schema_nm")

In [0]:
now = datetime.now()
file_arrival_time = now.strftime('%Y-%m-%d %H:%M:%S')
excn_id = f"{now.year}{now.month}{now.day}{now.hour}{now.minute}{now.second}{random.randint(101,999)}"
file_name = file_key.split("/")[-1]

In [0]:
print(bucket)
print(file_key)

In [0]:
matched = None
for row in spark.sql(f"SELECT * FROM {catalog}.{schema_nm}.dc_entity_mstr").collect():
	prefix = strip_glob(row.file_nm)
	if prefix and prefix in file_name:
		if matched is None or len(strip_glob(matched.file_nm)) < len(prefix):
			matched = row
if matched is None:
	dbutils.jobs.taskValues.set("data_found" , "NO")
	raise Exception(f"No dc_entity_mstr entry matches '{file_name}'")  # legacy hard-stop

In [0]:
# publish for downstream tasks + record arrival
tv = dbutils.jobs.taskValues
task_values = {
    "data_found": "YES",
    "bucket": bucket,
    "file_key": file_key,
    "app_id": matched.app_id,
    "file_name": file_name,
    "entity_id": matched.entity_id,
    "excn_id": excn_id,
    "entity_name": matched.entity_nm,
    "catalog": catalog,
    "schema_nm": schema_nm,
    "service_type": matched.service,
    "pre_contract_path": matched.raw_pre_processing_path,
    "pre_contract_file_path": f"s3://{matched.raw_pre_processing_path}/{file_name}",
    "post_process_path": matched.raw_post_processing_path,
    "file_arrival_time": file_arrival_time
}
for key, val in task_values.items():
    tv.set(key=key, value=val)
if matched.service == "INF":
    tv.set(key="param_file_path", value=matched.param_file_path)